## Imports

In [2]:
import os
import glob

import astropy.io.fits as fits
from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm
from astropy.io import ascii

import numpy as np
import matplotlib.pyplot as plt

from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture
import pandas as pd
from jwst import datamodels
from stdatamodels.jwst.datamodels import MiriWFSSSpecwcsModel
import cv2
from scipy.optimize import curve_fit
from jwst.assign_wcs import util

ModuleNotFoundError: No module named 'pandas'

In [2]:
input_dir = '/Users/morrison/MIRI_WFSS/Data4Jane/'

input_specwcs = 'MIRI_WFSS_specwcs_20250911.asdf'
tracefile = 'NewTraceList_20250901.txt'
#tracefile = 	'trace_ver2.txt'

In [3]:
# and plot the trace file pick the trace for x = 450, y = 450
df = pd.read_csv(tracefile, delimiter=" ")

x = df["X0"].values
y = df["Y0"].values
dx = df["dx"].values
dy = df["dy"].values
lam = df["lam"].values

# Filter the subset
x0 = 450
y0 = 450
subset = df[(df['X0'] == x0) & (df['Y0'] == y0)]
#xpos = subset['X0'] + subset['dx']
lam = subset['lam']
dx = subset['dx']
dy = subset['dy']
nvalues = len(dx)

In [4]:
# read in the datamodel
model =  MiriWFSSSpecwcsModel(input_specwcs)

In [5]:
# pull out polynomials from datamodel
xmodel = model.dispx
ymodel = model.dispy
lmodel = model.displ
inv_lmodel = model.invdispl
orders = model.orders

print('xmodel', xmodel, type(xmodel))
print('indispl',inv_lmodel)
print('lmodel', lmodel, type(lmodel))
print('orders', orders, type(orders))
print(ymodel[0])
print(ymodel[0][0])

for sub_x in xmodel:
    print(sub_x)
for sub_y in ymodel:
    print('SIZE OF dispy', len(sub_y))

xmodel [[<Polynomial2D(2, c0_0=0.25132305, c1_0=-0., c2_0=0., c0_1=-0., c0_2=0., c1_1=0.)>, <Polynomial2D(2, c0_0=-0.81221366, c1_0=0.00000001, c2_0=-0., c0_1=0., c0_2=-0., c1_1=-0.)>, <Polynomial2D(2, c0_0=2.24792305, c1_0=-0.00000001, c2_0=0., c0_1=-0., c0_2=0., c1_1=0.)>, <Polynomial2D(2, c0_0=-3.46016159, c1_0=0.00000001, c2_0=-0., c0_1=0., c0_2=-0., c1_1=-0.)>]] <class 'stdatamodels.properties.ListNode'>
indispl [<Polynomial1D(1, c0=-0.28688148, c1=0.09180207)>]
lmodel [<Polynomial1D(1, c0=3.125, c1=10.893)>] <class 'stdatamodels.properties.ListNode'>
orders [1] <class 'stdatamodels.properties.ListNode'>
[<Polynomial2D(2, c0_0=97.10928518, c1_0=0.00000296, c2_0=-0., c0_1=-0.00000011, c0_2=0., c1_1=0.)>, <Polynomial2D(2, c0_0=-78.4999802, c1_0=-0.00002318, c2_0=0.00000002, c0_1=0.00000058, c0_2=-0., c1_1=-0.)>, <Polynomial2D(2, c0_0=-235.05338959, c1_0=0.00004714, c2_0=-0.00000004, c0_1=-0.00000121, c0_2=0., c1_1=0.)>, <Polynomial2D(2, c0_0=-75.48364077, c1_0=-0.00002762, c2_0=0.00

In [6]:
x0 = 450
y0 = 450


In [7]:
# USE THIS TO read in asdf WFSS SPECWCS
def fit_2D26_22(coords, *vars):
    """
    From niriss_specwcs code:
    https://grit.stsci.edu/NIRISS/reference-file-creation/specwcs/-/blob/main/niriss_specwcs/make_specwcs.py?ref_type=heads#L433
    """
    # convert the given list of 2D models
    e_polynomials = vars
    e = []
    for poly in e_polynomials:
        e.extend([
            poly.c0_0.value,
            poly.c1_0.value,
            poly.c0_1.value,
            poly.c2_0.value,
            poly.c1_1.value,
            poly.c0_2.value
    ])
    
    # Generalized 2D polynomial (2,n) order
    (x,y,t) = coords
  
    n = len(e)//6
    f = 0
   
    for i in range(n):
        f = f + t**i * (e[i*6] + x*e[i*6+1] + y*e[i*6+2] + x**2*e[i*6+3] + x*y*e[i*6+4] + y**2*e[i*6+5])
    return f

In [ ]:
# This one does not work with the asdf coefficients. THis one works if using the conf file to read in the coefficients

def fit_2D26_2(coords, *vars):
    """
    From niriss_specwcs code:
    https://grit.stsci.edu/NIRISS/reference-file-creation/specwcs/-/blob/main/niriss_specwcs/make_specwcs.py?ref_type=heads#L433
    """
    # Generalized 2D polynomial (2,n) order
    (x,y,t) = coords
    e = vars
    n = len(e)//6
    f = 0
    print('vars',vars)

    for i in range(n):
        f = f + t**i * (e[i*6] + x*e[i*6+1] + y*e[i*6+2] + x**2*e[i*6+3] + x*y*e[i*6+4] + y**2*e[i*6+5])
    return f

In [ ]:

# Test mapping from direct to dispersed image (BACKWARD)

this_order = 1
order_mapping = {int(k): v for v, k in enumerate(orders)}
print('order mapping',order_mapping)
iorder = order_mapping[this_order]

#pull out first list
lmodel2 = lmodel
ymodel2 = ymodel[iorder]
xmodel2 = xmodel[iorder]
lmodel2 = lmodel[iorder]
inv_lmodel2 = inv_lmodel[iorder]
print('ymodel2',ymodel2) 
print('lmodel2', lmodel2)
print('inv_lmodel2',inv_lmodel2)

lmin = lmodel2.c0
lmax = lmodel2.c0 + lmodel2.c1
print('nvalues', nvalues)
dx2 = np.zeros(nvalues)
dy_fitted = np.zeros(nvalues)
dy2 = np.zeros(nvalues)
ts2 = inv_lmodel2(lam)

print('model t', ts2[0:30], ts2[-30:])

result_x = fit_2D26_22((subset["X0"].values, subset["Y0"].values, ts2), *xmodel2)
result_y = fit_2D26_22((subset["X0"].values, subset["Y0"].values, ts2), *ymodel2)



In [ ]:
fig = plt.figure(figsize= (10,10))
ax1 =fig.add_subplot(1,2,1)
ax2 =fig.add_subplot(1,2,2)

ax1.set_xlabel('Lambda')
ax1.set_ylabel('dy', color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax2.set_xlabel('Lambda')
ax2.set_ylabel('dx', color='b')
ax2.tick_params(axis='y', labelcolor='b')


print(lam.shape, dy.shape, dy2.shape)
ax1.plot(lam, dy, label=" dy from trace", color='purple', linewidth=2, linestyle='solid')

ax1.plot(lam, result_y, label="Fitted dy", color='red', linewidth=2, linestyle='dotted')

ax2.plot(lam, dx, label=" dx from trace", color='purple', linewidth=2, linestyle='solid')

ax2.plot(lam, result_x, label="Fitted dx", color='red', linewidth=2, linestyle='dotted')


ax1.legend()
ax2.legend()

In [ ]:
fig = plt.figure(figsize= (8,8))
ax1 =fig.add_subplot(1,1,1)


ax1.set_xlabel('dx')
ax1.set_ylabel('dy', color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax1.plot(result_x, result_y, label=" Values from trace file ", color='purple', linewidth=2, linestyle='solid')

ax1.plot(dx, dy, label="Fitted values", color='red', linewidth=2, linestyle='dotted')

ax1.legend()
